In [10]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# ==========================================
# 1. Load Dataset
# ==========================================
try:
    df = pd.read_csv('Telco_Customer_Churn.csv')
except FileNotFoundError:
    # If it's saved as an Excel file, I'll use read_excel
    df = pd.read_excel('Telco_Customer_Churn.xlsx')

# ==========================================
# 2. Data Cleaning & Quality Assurance (QA)
# ==========================================
# TotalCharges often has hidden spaces
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
imputer = SimpleImputer(strategy='median')
df['TotalCharges'] = imputer.fit_transform(df[['TotalCharges']])

# ==========================================
# 3. Professional Feature Engineering
# ==========================================
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                'TechSupport', 'StreamingTV', 'StreamingMovies']
# Ensure columns exist before summing
df['Service_Intensity'] = (df[service_cols] == 'Yes').sum(axis=1)

df['Contract_Rank'] = df['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
df['Value_at_Risk'] = df['MonthlyCharges'] * df['tenure']

# ==========================================
# 4. Target Mapping & ID Preservation
# ==========================================
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
business_key = df[['customerID', 'Churn', 'Value_at_Risk']].copy()
business_key.to_csv('Customer_Retention_Key.csv', index=False)

# ==========================================
# 5. Final Preprocessing for Logan
# ==========================================
df_modeling = df.drop(['customerID', 'Contract'], axis=1)
df_final = pd.get_dummies(df_modeling, drop_first=True)

# ==========================================
# 6. Save Processed Data
# ==========================================
df_final.to_csv('Telco_Cleaned_Model_Ready.csv', index=False)

print("--- Professional Data Engineering Complete ---")

--- Professional Data Engineering Complete ---
Successfully processed the file from your Project_2_customer_churn folder!


# Phase 1: Advanced Data Engineering & Preprocessing

---

### **Overview**
This phase transforms the raw **IBM Telco Customer Churn** dataset into a high-signal, model-ready format. The pipeline is engineered to support a comparative study between **Random Forest** and **XGBoost** architectures while ensuring the final results are actionable for business stakeholders.

---

### **1. Data Integrity & Quality Assurance (QA)**
* **Type Correction (Numerical Cleaning)**
    * **The Action:** Coerced `TotalCharges` into numeric floats via `pd.to_numeric`.
    * **The Why:** New customers often have blank spaces in their billing data. Without this step, Python treats the whole column as an object (text), making mathematical operations impossible.
* **Robust Median Imputation**
    * **The Action:** Utilized `SimpleImputer` with a **median strategy**.
    * **The Why:** Algorithms like **Random Forest** cannot process null values (`NaN`). The median was selected as it is statistically resistant to outliers, providing a realistic representative value for missing entries without skewing the distribution.
* **Billing Logic Audit**
    * **The Action:** Implemented a QA check to flag records where `TotalCharges` were mathematically inconsistent with `MonthlyCharges`.
    * **The Why:** Ensures the models are trained on clean, verified data points rather than corrupted or logically impossible records.

---

### **2. Sophisticated Feature Engineering**
To extract maximum predictive power from categorical data, three **High-Signal** features were synthesized:
* **Service Intensity Score**
    * **The Action:** Aggregated six binary service indicators into a single cumulative engagement metric.
    * **The Why:** Quantifies **ecosystem stickiness**. Historically, customers with higher service density exhibit significantly lower churn rates due to increased switching costs and deeper product integration.
* **Contractual Ordinal Ranking**
    * **The Action:** Mapped contract types to a **0-2 ordinal scale**.
    * **The Why:** Standard one-hot encoding fails to capture the natural hierarchy of commitment. Ordinal mapping ensures the model recognizes the increased stability of long-term contracts (**2-year > 1-year > Month-to-month**).
* **Value-at-Risk (VaR)**
    * **The Action:** Calculated a financial proxy utilizing the product of `MonthlyCharges` and `tenure`.
    * **The Why:** Shifts the focus from **Churn Probability** to **Financial Impact**. This enables stakeholders to prioritize retention efforts based on potential revenue loss rather than just raw percentage points.

---

### **3. Data Governance & Model Optimization**
* **Decoupled Business Keys**
    * **The Action:** Isolated `customerID` from the training set while preserving a standalone `Customer_Retention_Key.csv`.
    * **The Why:** Prevents **Overfitting** (the model memorizing unique identifiers) while maintaining the ability to map model insights back to specific accounts for targeted intervention.
* **Multicollinearity Management**
    * **The Action:** Implemented One-Hot Encoding via `pd.get_dummies` with the `drop_first=True` parameter.
    * **The Why:** Mitigates the **Dummy Variable Trap**, ensuring mathematical stability within the feature matrix by removing redundant information that can degrade model interpretability.

---

### **4. Pipeline Deliverables**
The resulting **100% numeric matrix** is optimized for high-performance ensemble learning:
1. **Random Forest Compatibility:** Full imputation ensures stable node splitting and handles non-linear relationships.
2. **XGBoost Compatibility:** Binary target mapping (**1/0**) and high-density feature engineering support efficient gradient boosting operations and loss function optimization.